# Stage 4: Machine Learning Recommendation Models (Beginner's Guide)
### E-Commerce Intelligence & Decision Support Platform

---

### Why use Machine Learning?
The **Popularity Baseline** gives the exact same 5 bestsellers to every customer.
- But Customer A might only buy **baking supplies**.
- Customer B might only buy **Christmas decorations**.

Machine Learning allows us to **personalize** recommendations:
1. For each customer and each candidate product, we predict a **purchase probability** (between 0% and 100%).
2. We rank all products by probability for that specific customer.
3. We recommend their **Top 5 highest-probability items**.

We will train two beginner-friendly models:
- **Logistic Regression:** Simple, fast, linear probability model.
- **Decision Tree:** Intuitive if-then decision rules.


### Step 1: Import Libraries and Load Features

In [1]:
import pandas as pd
import numpy as np
import os
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, recall_score, roc_auc_score

# Helper to find data folder
def get_path(fname):
    p1 = os.path.join("data", "processed", fname)
    return p1 if os.path.exists(p1) else os.path.join("..", "data", "processed", fname)

cust_feat = pd.read_csv(get_path("customer_features.csv"))
prod_feat = pd.read_csv(get_path("product_features.csv"))
cp_feat = pd.read_csv(get_path("customer_product_features.csv"))
historical = pd.read_csv(get_path("historical_transactions.csv"), dtype={"InvoiceNo": str, "StockCode": str})
evaluation = pd.read_csv(get_path("evaluation_transactions.csv"), dtype={"InvoiceNo": str, "StockCode": str})

print("All feature tables loaded successfully!")


All feature tables loaded successfully!


### Step 2: Build the Candidate Dataset
We take the **top 500 popular products** and all **2,602 eligible customers**.
This gives $2,602 \times 500 = 1,301,000$ possible customer-product pairs.
- Target label `purchased = 1`: If the customer actually bought this product in evaluation.
- Target label `purchased = 0`: If they did not buy it.


In [2]:
# 1. Eligible customers & top 500 products
eligible_custs = sorted(set(historical["CustomerID"].unique()) & set(evaluation["CustomerID"].unique()))
top_500_prods = historical.groupby("StockCode")["InvoiceNo"].nunique().sort_values(ascending=False).head(500).index.tolist()

# 2. Candidate pairs
pairs = pd.DataFrame([(c, p) for c in eligible_custs for p in top_500_prods], columns=["CustomerID", "StockCode"])

# 3. Ground truth labels from evaluation
eval_buys = evaluation[evaluation["CustomerID"].isin(eligible_custs)].drop_duplicates(subset=["CustomerID", "StockCode"])[["CustomerID", "StockCode"]].copy()
eval_buys["purchased"] = 1
pairs = pairs.merge(eval_buys, on=["CustomerID", "StockCode"], how="left")
pairs["purchased"] = pairs["purchased"].fillna(0).astype(int)

# 4. Merge customer features
cust_cols = ["CustomerID", "total_orders", "total_spending", "total_quantity", "unique_products", "avg_order_value", "recency_days", "tenure_days"]
pairs = pairs.merge(cust_feat[cust_cols], on="CustomerID", how="left")

# 5. Merge product features
prod_cols = ["StockCode", "total_quantity_sold", "total_revenue", "total_orders", "unique_customers", "avg_unit_price", "popularity_score"]
pairs = pairs.merge(prod_feat[prod_cols].rename(columns={"total_orders": "prod_orders"}), on="StockCode", how="left")

# 6. Merge interaction features (past history with this product)
cp_cols = ["CustomerID", "StockCode", "purchase_count", "total_qty_purchased", "days_since_last_purchase", "has_previous_purchase"]
pairs = pairs.merge(cp_feat[cp_cols], on=["CustomerID", "StockCode"], how="left").fillna(0)

print(f"ML Dataset built: {pairs.shape[0]:,} rows, {pairs.shape[1]} columns")
print(f"Purchased = 1: {pairs['purchased'].sum():,} ({pairs['purchased'].mean()*100:.2f}%)")
pairs.head(3)


ML Dataset built: 1,301,000 rows, 20 columns
Purchased = 1: 54,359 (4.18%)


,CustomerID,StockCode,purchased,total_orders,total_spending,total_quantity,unique_products,avg_order_value,recency_days,tenure_days,total_quantity_sold,total_revenue,prod_orders,unique_customers,avg_unit_price,popularity_score,purchase_count,total_qty_purchased,days_since_last_purchase,has_previous_purchase
0,12347,85123A,0,4,2434.96,1626,90,608.74,54,212,75435,201677.23,3916,1313,2.86,100.0,0.0,0.0,0.0,0.0
1,12347,22423,1,4,2434.96,1626,90,608.74,54,212,18478,212307.90,2464,1061,12.42,100.0,3.0,9.0,54.0,1.0
2,12347,85099B,0,4,2434.96,1626,90,608.74,54,212,65615,115456.96,2252,785,1.92,100.0,0.0,0.0,0.0,0.0


### Step 3: Train / Test Split by Customer
We split **by Customer** (not randomly by row) so the model is evaluated on customers it has never seen during training:
- **70% Train Customers** (1,821 customers $\rightarrow$ 910,500 rows)
- **30% Test Customers** (781 customers $\rightarrow$ 390,500 rows)


In [3]:
train_custs, test_custs = train_test_split(eligible_custs, test_size=0.30, random_state=42)

train_df = pairs[pairs["CustomerID"].isin(train_custs)].copy()
test_df = pairs[pairs["CustomerID"].isin(test_custs)].copy()

feature_cols = [c for c in pairs.columns if c not in ["CustomerID", "StockCode", "purchased"]]

X_train = train_df[feature_cols].values
y_train = train_df["purchased"].values

X_test = test_df[feature_cols].values
y_test = test_df["purchased"].values

# Standardize features (mean=0, std=1)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Features used ({len(feature_cols)}): {feature_cols}")
print(f"Train samples: {X_train.shape[0]:,} | Test samples: {X_test.shape[0]:,}")


Features used (17): ['total_orders', 'total_spending', 'total_quantity', 'unique_products', 'avg_order_value', 'recency_days', 'tenure_days', 'total_quantity_sold', 'total_revenue', 'prod_orders', 'unique_customers', 'avg_unit_price', 'popularity_score', 'purchase_count', 'total_qty_purchased', 'days_since_last_purchase', 'has_previous_purchase']
Train samples: 910,500 | Test samples: 390,500


### Step 4: Train Logistic Regression Model
Logistic Regression fits a smooth probability curve.
We use `class_weight='balanced'` because only ~4% of pairs are purchases.

In [4]:
lr_model = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)
lr_model.fit(X_train_scaled, y_train)

# Predict purchase probabilities on test cohort
test_df["lr_prob"] = lr_model.predict_proba(X_test_scaled)[:, 1]

print(f"Logistic Regression ROC-AUC Score: {roc_auc_score(y_test, test_df['lr_prob']):.4f}")


Logistic Regression ROC-AUC Score: 0.8200


### Step 5: Train Decision Tree Model
A Decision Tree learns simple if-else decision rules. We set `max_depth=10` to prevent overfitting.

In [5]:
dt_model = DecisionTreeClassifier(max_depth=10, min_samples_leaf=50, class_weight="balanced", random_state=42)
dt_model.fit(X_train_scaled, y_train)

# Predict purchase probabilities on test cohort
test_df["dt_prob"] = dt_model.predict_proba(X_test_scaled)[:, 1]

print(f"Decision Tree ROC-AUC Score: {roc_auc_score(y_test, test_df['dt_prob']):.4f}")


Decision Tree ROC-AUC Score: 0.8099


### Step 6: Generate Personalized Top-5 Recommendations & Evaluate
For each test customer, we take the 5 products with the highest predicted probability and check if any were actually bought!

In [6]:
# Ground truth for test customers
test_eval = evaluation[evaluation["CustomerID"].isin(test_custs)]
test_ground_truth = test_eval.groupby("CustomerID")["StockCode"].apply(set).to_dict()

# Evaluate both models
def evaluate_ml_model(prob_col, model_name):
    hits = 0
    for cust_id in test_custs:
        cust_rows = test_df[test_df["CustomerID"] == cust_id].sort_values(by=prob_col, ascending=False)
        top_5_recs = cust_rows.head(5)["StockCode"].tolist()
        actual = test_ground_truth.get(cust_id, set())
        if set(top_5_recs) & actual:
            hits += 1
    hit_rate = (hits / len(test_custs)) * 100
    print(f"{model_name} Hit-Rate@5: {hit_rate:.2f}% ({hits}/{len(test_custs)} test customers)")
    return hit_rate, hits

lr_hr, lr_hits = evaluate_ml_model("lr_prob", "Logistic Regression")
dt_hr, dt_hits = evaluate_ml_model("dt_prob", "Decision Tree")


Logistic Regression Hit-Rate@5: 75.03% (586/781 test customers)


Decision Tree Hit-Rate@5: 76.31% (596/781 test customers)


### Step 7: Inspect Sample Personalized Recommendations
Let's look at the personalized recommendations for one specific customer:

In [7]:
sample_id = test_custs[0]
sample_recs = test_df[test_df["CustomerID"] == sample_id].sort_values(by="dt_prob", ascending=False).head(5)

print(f"Top-5 Recommendations for Customer {sample_id}:")
for rank, (_, row) in enumerate(sample_recs.iterrows(), 1):
    desc = prod_feat[prod_feat["StockCode"] == row["StockCode"]]["description"].values[0]
    bought_eval = "YES [HIT]" if row["StockCode"] in test_ground_truth.get(sample_id, set()) else "NO"
    print(f"  #{rank}: {row['StockCode']} | Prob: {row['dt_prob']:.3f} | Bought in Evaluation? {bought_eval} | {desc}")


Top-5 Recommendations for Customer 16128:
  #1: 85152 | Prob: 0.944 | Bought in Evaluation? NO | HAND OVER THE CHOCOLATE   SIGN 
  #2: 22111 | Prob: 0.944 | Bought in Evaluation? NO | SCOTTIE DOG HOT WATER BOTTLE
  #3: 22197 | Prob: 0.944 | Bought in Evaluation? NO | SMALL POPCORN HOLDER
  #4: 21754 | Prob: 0.944 | Bought in Evaluation? NO | HOME BUILDING BLOCK WORD
  #5: 22507 | Prob: 0.908 | Bought in Evaluation? NO | MEMO BOARD RETROSPOT  DESIGN
